# Reproduction of Domainnome Analysis

## Step 1 - Variant Extraction (Core)

In [1]:
import pandas as pd
from py2neo import Graph
import re
import os

URI = "bolt://localhost:7687"
PWD = os.environ.get("NEO4J_PWD", "multiomicskg")
DBNAME = os.environ.get("NEO4J_DATABASE", "neo4j")
CLINICAL_VARIANTS_FROM_UNIPROT_DOMAINONE_FILE = "/Users/julian/datastore/domainome_local/analysis_files/clinical_variants_uniprot_proteinAPI_domainomegenes.txt"
GNOMAD_VARIANTS_FROM_DOMAINOME_FILE = "/Users/julian/datastore/domainome/analysis_files/gnomad_v4_variants_domainome.txt"
CLINICAL_VARIANTS_FROM_MULTIOMICSKB_FILE = "02a-clinical_variants_uniprot_proteinAPI_domainomegenes_from_multiomicskb_core.csv"
CLINVAR_VARIANTS_FROM_MULTIOMICSKB_FILE = "02b-variant_summary_from_multiomicskb_core.tsv"
GNOMAD_VARIANTS_FROM_MULTIOMICSKB_FILE = "02c-gnomad_v2_variants_from_multiomicskb_core.tsv"

graph = Graph(URI, auth=("neo4j", PWD), name=DBNAME)

### Retrieve Variant Data

In [2]:
# Get list of selected Domainome UniProt IDs
selected_uniprot_ids = []
variants_uniprot_domainome_df = pd.DataFrame()
with open(CLINICAL_VARIANTS_FROM_UNIPROT_DOMAINONE_FILE, "r") as f:
    variants_uniprot_domainome_df = pd.read_csv(f, sep="\t", names=["uniprot_ID","uniprot_entry","gene_name","sequence","type","begin","end","wt_aa","mut_aa","genomic_location","codon","consequence","source","str(somatic_status)","clinvar_id","exac_id","topmed_id","gnomad_id","NCITCGA_id","NCITCGA_cosmic_id","clingen_id","dbSNP_id","ensembl_id","str(sift_score)","sift_class","str(polyphen_score)","polyphen_class","clinvar_class","ensembl_class","uniprot_class","NCITCGA_class","descr"])
    selected_uniprot_ids = variants_uniprot_domainome_df['uniprot_ID'].unique().tolist()

/var/folders/cy/0dgyz7x96x7d8d3619f5j45w0000gn/T/ipykernel_51621/3830643012.py:5: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  variants_uniprot_domainome_df = pd.read_csv(f, sep="\t", names=["uniprot_ID","uniprot_entry","gene_name","sequence","type","begin","end","wt_aa","mut_aa","genomic_location","codon","consequence","source","str(somatic_status)","clinvar_id","exac_id","topmed_id","gnomad_id","NCITCGA_id","NCITCGA_cosmic_id","clingen_id","dbSNP_id","ensembl_id","str(sift_score)","sift_class","str(polyphen_score)","polyphen_class","clinvar_class","ensembl_class","uniprot_class","NCITCGA_class","descr"])


In [3]:
variants_uniprot_domainome_df["clinvar_class"].unique().tolist()

[nan,
 'Pathogenic',
 'Variant of uncertain significance',
 'Benign',
 'Likely benign',
 'Likely pathogenic',
 'Conflicting interpretations of pathogenicity',
 'Risk factor',
 'Drug response',
 'association']

In [4]:
# Query Neo4j for all variants associated with the selected UniProt IDs,
# including those without ClinVar significance or with a single ClinVar significance from the list
query = """
MATCH (g:gene)-[:reads]->(t:transcript)-[:generates]->(p:protein)-[r]-(s:snv)-[:maps_res]->(res:residue)<-[:has]-(m:model)
WHERE p.accession IN $selected_uniprot_ids AND (s.clinvar_significance IS NULL OR s.clinvar_significance IN ["Benign", "Conflicting_classifications_of_pathogenicity", "Likely_benign", "Likely_pathogenic", "Pathogenic", "risk_factor", "Uncertain_significance" ]) AND res.location="core"
WITH replace(replace(replace(s.clinvar_significance, "Uncertain_significance", "Variant of uncertain significance"), "Conflicting_classifications_of_pathogenicity", "Conflicting interpretations of pathogenicity"), "_", " ") as clinvar_significance, g, t, p, s
RETURN p.accession as uniprot_ID, p.name as uniprot_entry, g.name as gene_name, p.sequence as sequence, "VARIANT" as type, s.protein_position as begin, s.protein_position as end, substring(s.amino_acids,0,1) as wt_aa, substring(s.amino_acids,2,3) as mut_aa, split(s.id,"_")[0] as genomic_location, upper(s.codons) as codon, "missense" as consequence, "N/A" as source, "N/A, need decoding" as somatic_status, "N/A need splitting" as clinvar_id, "N/A" as exac_id, "N/A" as topmed_id,"N/A need splitting" as gnomad_id, "N/A" as NCITCGA_id,"N/A" as NCITCGA_cosmic_id, "N/A" as clingen_id, "N/A" as dbSNP_id, "N/A" as ensembl_id, s.sift as sift_score, replace(replace(s.sift_significance, "_low", " - low"),"_", " ") as sift_class, s.polyphen as polyphen_score, replace(s.polyphen_significance, "_", " ") as polyphen_class, toUpper(substring(clinvar_significance, 0, 1)) + substring(clinvar_significance, 1) as clinvar_class, "N/A" as ensembl_class, "N/A" as uniprot_class, "N/A" as NCITCGA_class, "N/A" as descr
ORDER BY s.protein_position
"""
variants_df = graph.run(query, selected_uniprot_ids=selected_uniprot_ids).to_data_frame()

# 4014423 rows in total, 4014423 unique variants, 32 variables (all clinvar_significance)
# 11660 rows in total, 32 variables (only single clinvar_significance)
# 126870 rows in total, 32 variables (only single clinvar_significance or no clinvar_significance)

In [5]:
variants_df["clinvar_class"].unique().tolist()

[None,
 'Variant of uncertain significance',
 'Pathogenic',
 'Likely pathogenic',
 'Conflicting interpretations of pathogenicity',
 'Likely benign',
 'Benign']

In [6]:
variants_df.to_csv(CLINICAL_VARIANTS_FROM_MULTIOMICSKB_FILE, sep='\t', header=False, index=False)

In [7]:
# Query Neo4j for all Clinvar variants
query = """
MATCH (g:gene)-[:reads]->(t:transcript)-[:generates]->(p:protein)-[r]-(s:snv)-[:maps_res]->(res:residue)<-[:has]-(m:model)
WHERE s.source CONTAINS "clinvar" AND res.location="core"
WITH replace(s.codons, "/", ">") as mutation, g, t, p, s
RETURN "N/A" as AlleleID, "single nucleotide variant" as Type, "NM_X.X" as Name, "N/A" as GeneID, g.name as GeneSymbol, "N/A" as HGNC_ID, s.clinvar_significance as ClinicalSignificance, s.CDS_position as CDSPosition, mutation as Mutation, s.amino_acids as AminoAcids, s.protein_position as ProteinPosition, s.mane_select as ManeSelect
"""
clinvar_variants_df = graph.run(query, selected_uniprot_ids=selected_uniprot_ids).to_data_frame()

In [8]:
# make string changes to satisfy the format of variant_summary_jan24_SNVs_coding

# remove lowercase portion of mutation to reduce to only the nucleotide change (e.g., "T>G" instead of "Tgc>Ggc")
clinvar_variants_df['Mutation'] = clinvar_variants_df['Mutation'].str.replace("[a-z]", "", regex=True)

# to change single letter amino acid codes to three letter codes
aa_dict = {
    'A': 'Ala', 'C': 'Cys', 'D': 'Asp', 'E': 'Glu', 'F': 'Phe',
    'G': 'Gly', 'H': 'His', 'I': 'Ile', 'K': 'Lys', 'L': 'Leu',
    'M': 'Met', 'N': 'Asn', 'P': 'Pro', 'Q': 'Gln', 'R': 'Arg',
    'S': 'Ser', 'T': 'Thr', 'V': 'Val', 'W': 'Trp', 'Y': 'Tyr',
    'U': 'Sec', 'O': 'Pyl', '*': 'Ter'
}

# change categories of clinvar significance
clinvar_significance_dict = {
    "Benign": "Benign",
    "Likely_benign": "Likely benign",
    "Benign/Likely_benign": "Benign/Likely benign",
    "Pathogenic": "Pathogenic",
    "Pathogenic/Likely pathogenic": "Pathogenic/Likely pathogenic",
    "Likely_pathogenic": "Likely pathogenic",
    "Conflicting_classifications_of_pathogenicity": "Conflicting interpretations of pathogenicity",
    "risk_factor": "Risk factor",
    "Uncertain_significance": "Uncertain significance"
}
for k, v in clinvar_significance_dict.items():
    clinvar_variants_df['ClinicalSignificance'] = clinvar_variants_df['ClinicalSignificance'].str.replace(k, v, regex=True)

clinvar_variants_df['ClinicalSignificance'] = clinvar_variants_df['ClinicalSignificance'].str.replace(",", "; ", regex=False)
clinvar_variants_df['Name'] = (
    clinvar_variants_df['ManeSelect'] + "("
    + clinvar_variants_df['GeneSymbol'] + "):c."
    + clinvar_variants_df['CDSPosition'].astype(str)
    + clinvar_variants_df['Mutation']
    + " (p." 
    + clinvar_variants_df['AminoAcids'].str[0].map(aa_dict)
    + clinvar_variants_df['ProteinPosition'].astype(str)
    + clinvar_variants_df['AminoAcids'].str[2].map(aa_dict)
    + ")"
)

In [9]:
clinvar_variants_df.to_csv(CLINVAR_VARIANTS_FROM_MULTIOMICSKB_FILE, sep='\t', header=False, index=False)

In [10]:
# selected_genes = []
# with open("/Users/julian/datastore/domainome/analysis_files/variant_summary_jan24_SNVs_coding.txt", "r") as f:
#     df = pd.read_csv(f, sep="\t", low_memory=False, names=["#AlleleID","Type","Name","GeneID","GeneSymbol","HGNC_ID","ClinicalSignificance","ClinSigSimple","LastEvaluated","RS# (dbSNP)","nsv/esv (dbVar)","RCVaccession","PhenotypeIDS","PhenotypeList","Origin","OriginSimple","Assembly","ChromosomeAccession","Chromosome","Start","Stop","ReferenceAllele","AlternateAllele","Cytogenetic","ReviewStatus","NumberSubmitters","Guidelines","TestedInGTR","OtherIDs","SubmitterCategories","VariationID","PositionVCF","ReferenceAlleleVCF","AlternateAlleleVCF","SomaticClinicalImpact","SomaticClinicalImpactLastEvaluated","ReviewStatusClinicalImpact","Oncogenicity","OncogenicityLastEvaluated","ReviewStatusOncogenicity","SCVsForAggregateGermlineClassification","SCVsForAggregateSomaticClinicalImpact","SCVsForAggregateOncogenicityClassification"])
#     selected_genes = df['GeneSymbol'].unique().tolist()

# print(f"Number of unique genes in ClinVar variants: {len(selected_genes)}")

# # Number of unique genes in ClinVar variants: 19184
clinvar_variants_df['ClinicalSignificance'].value_counts().to_dict()

{'Uncertain significance': 259017,
 'Conflicting interpretations of pathogenicity': 11270,
 'Likely pathogenic': 10900,
 'Pathogenic': 7480,
 'Likely benign': 5958,
 'Pathogenic/Likely pathogenic': 4338,
 'Benign': 1869,
 'Benign/Likely benign': 829,
 'not_provided': 679,
 'drug_response': 99,
 'other': 96,
 'no_classification_for_the_single_variant': 71,
 'Likely pathogenic; _low_penetrance': 27,
 'Risk factor': 26,
 'Affects': 25,
 'Uncertain significance/Uncertain_risk_allele': 24,
 'Pathogenic|other': 19,
 'association': 16,
 'no_classifications_from_unflagged_records': 14,
 'Likely_risk_allele': 11,
 'Likely pathogenic/Likely_risk_allele': 5,
 'Benign/Likely benign|other': 4,
 'Conflicting interpretations of pathogenicity|association': 4,
 'Pathogenic|drug_response': 3,
 'Conflicting interpretations of pathogenicity|other': 3,
 'Uncertain_risk_allele': 3,
 'Likely benign|other': 3,
 'VUS-high': 2,
 'Pathogenic/Likely pathogenic|other': 2,
 'Uncertain significance/VUS-high': 2,
 'p

In [11]:
# List unique clinvar significances from original clinvar domainome variants file
variants_clinvar_domainome_df = pd.DataFrame()
with open("/Users/julian/datastore/domainome/analysis_files/variant_summary_jan24_SNVs_coding.txt", "r") as f:
    variants_clinvar_domainome_df = pd.read_csv(f, sep="\t", names=["#AlleleID","Type","Name","GeneID","GeneSymbol","HGNC_ID","ClinicalSignificance","ClinSigSimple","LastEvaluated","RS# (dbSNP)","nsv/esv (dbVar)","RCVaccession","PhenotypeIDS","PhenotypeList","Origin","OriginSimple","Assembly","ChromosomeAccession","Chromosome","Start","Stop","ReferenceAllele","AlternateAllele","Cytogenetic","ReviewStatus","NumberSubmitters","Guidelines","TestedInGTR","OtherIDs","SubmitterCategories","VariationID","PositionVCF","ReferenceAlleleVCF","AlternateAlleleVCF","SomaticClinicalImpact","SomaticClinicalImpactLastEvaluated","ReviewStatusClinicalImpact","Oncogenicity","OncogenicityLastEvaluated","ReviewStatusOncogenicity","SCVsForAggregateGermlineClassification","SCVsForAggregateSomaticClinicalImpact","SCVsForAggregateOncogenicityClassification"])

variants_clinvar_domainome_df["ClinicalSignificance"].value_counts().to_dict()

/var/folders/cy/0dgyz7x96x7d8d3619f5j45w0000gn/T/ipykernel_51621/824970415.py:4: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  variants_clinvar_domainome_df = pd.read_csv(f, sep="\t", names=["#AlleleID","Type","Name","GeneID","GeneSymbol","HGNC_ID","ClinicalSignificance","ClinSigSimple","LastEvaluated","RS# (dbSNP)","nsv/esv (dbVar)","RCVaccession","PhenotypeIDS","PhenotypeList","Origin","OriginSimple","Assembly","ChromosomeAccession","Chromosome","Start","Stop","ReferenceAllele","AlternateAllele","Cytogenetic","ReviewStatus","NumberSubmitters","Guidelines","TestedInGTR","OtherIDs","SubmitterCategories","VariationID","PositionVCF","ReferenceAlleleVCF","AlternateAlleleVCF","SomaticClinicalImpact","SomaticClinicalImpactLastEvaluated","ReviewStatusClinicalImpact","Oncogenicity","OncogenicityLastEvaluated","ReviewStatusOncogenicity","SCVsForAggregateGermlineClassification","SCVsForAggregateSomaticClinicalImpact","SCVsForAggregateOnco

{'Uncertain significance': 1000328,
 'Likely benign': 417526,
 'Conflicting interpretations of pathogenicity': 88076,
 'Benign': 63830,
 'Pathogenic': 55853,
 'Likely pathogenic': 35027,
 'Benign/Likely benign': 28665,
 'Pathogenic/Likely pathogenic': 12673,
 'not provided': 6203,
 'other': 566,
 'drug response': 495,
 'no interpretation for the single variant': 436,
 'risk factor': 252,
 'association': 145,
 'Uncertain significance/Uncertain risk allele': 113,
 'Affects': 100,
 'Pathogenic; other': 68,
 'Likely risk allele': 64,
 'Pathogenic; drug response': 53,
 'protective': 22,
 'Conflicting interpretations of pathogenicity; other': 21,
 'Likely pathogenic/Likely risk allele': 18,
 'Uncertain risk allele': 16,
 'Likely pathogenic; drug response': 15,
 'Pathogenic; risk factor': 13,
 'Conflicting interpretations of pathogenicity; risk factor': 13,
 'Uncertain significance; drug response': 11,
 'Benign; other': 10,
 'Likely benign; other': 9,
 'Pathogenic/Likely pathogenic; other': 8

## Extract GnomAD data

In [12]:
# Get list of selected Domainome UniProt IDs
selected_gnomad_gene_ids = []
variants_gnomad_domainome_df = pd.DataFrame()
with open(GNOMAD_VARIANTS_FROM_DOMAINOME_FILE, "r") as f:
    variants_gnomad_domainome_df = pd.read_csv(f, sep="\t", names=["gene_id", "gene_name", "snv_id", "chr", "location", "base1", "base2", "snv_source", "consequence", "coding_change", "protein_change", "genome_af", "genome_allele_count", "genome_allele_number", "V15", "V16", "exome_af", "exome_allele_count", "exome_allele_number", "V20", "V21"])
    selected_gnomad_gene_ids = variants_gnomad_domainome_df['gene_id'].unique().tolist()

/var/folders/cy/0dgyz7x96x7d8d3619f5j45w0000gn/T/ipykernel_51621/2631071258.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  variants_gnomad_domainome_df = pd.read_csv(f, sep="\t", names=["gene_id", "gene_name", "snv_id", "chr", "location", "base1", "base2", "snv_source", "consequence", "coding_change", "protein_change", "genome_af", "genome_allele_count", "genome_allele_number", "V15", "V16", "exome_af", "exome_allele_count", "exome_allele_number", "V20", "V21"])


In [13]:
# Query Neo4j for all GnomAD variants
# note that the genome AF is set to zero so only the exome AF is used for the analysis
# likewise for the genome allele count, which is set to zero so only the exome allele count is used for the analysis
query = """
MATCH (g:gene)-[r1:reads]->(t:transcript)-[r2:generates]->(p:protein)-[r3]-(s:snv)-[:maps_res]->(res:residue)<-[:has]-(m:model)
WHERE g.id IN $selected_gnomad_gene_ids AND s.source CONTAINS "gnomad" AND res.location="core"
WITH replace(s.codons, "/", ">") as mutation,g,s
RETURN g.id as gene_id, g.name as gene_name, s.id as snv_id, "V4" as chr, s.location as location, "V6" as base1, "V7" as base2, s.source_id as snv_source, "missense_variant" as consequence, "c."+s.CDS_position+mutation as coding_change, "p."+substring(s.amino_acids,0,1)+s.protein_position+substring(s.amino_acids,2,3) as protein_change, 0 as genome_af, 0 as genome_allele_count, "GENOME_ALLELE_NUMBER" as genome_allele_number, "V15" as V15, "V16" as V16,s.gnomad_af as exome_af,1 as exome_allele_count, "EXOME_ALLELE_NUMBER" as exome_allele_number, "V20" as V20, "V21" as V21
"""
gnomad_variants_df = graph.run(query, selected_gnomad_gene_ids=selected_gnomad_gene_ids).to_data_frame()

In [14]:
# make string changes to satisfy the format of variant_summary_jan24_SNVs_coding

# remove lowercase portion of mutation to reduce to only the nucleotide change (e.g., "T>G" instead of "Tgc>Ggc")
gnomad_variants_df['coding_change'] = gnomad_variants_df['coding_change'].str.replace("[a-z]", "", regex=True)
gnomad_variants_df['coding_change'] = gnomad_variants_df['coding_change'].str.replace(".", "c.", regex=False)

# to change single letter amino acid codes to three letter codes
aa_dict = {
    'A': 'Ala', 'C': 'Cys', 'D': 'Asp', 'E': 'Glu', 'F': 'Phe',
    'G': 'Gly', 'H': 'His', 'I': 'Ile', 'K': 'Lys', 'L': 'Leu',
    'M': 'Met', 'N': 'Asn', 'P': 'Pro', 'Q': 'Gln', 'R': 'Arg',
    'S': 'Ser', 'T': 'Thr', 'V': 'Val', 'W': 'Trp', 'Y': 'Tyr',
    'U': 'Sec', 'O': 'Pyl', '*': 'Ter'
}
# gnomad_variants_df['protein_change'] = gnomad_variants_df['protein_change'].replace(aa_dict, regex=False, inplace=True)
pattern = re.compile("|".join(map(re.escape, aa_dict)))
gnomad_variants_df['protein_change'] = gnomad_variants_df['protein_change'].str.replace(pattern, lambda m: aa_dict[m.group(0)], regex=True)

In [15]:
gnomad_variants_df.to_csv(GNOMAD_VARIANTS_FROM_MULTIOMICSKB_FILE, sep='\t', header=False, index=False)